# BEP

# Pre-work

In [127]:
# imports

from pathlib import Path
import pandas as pd
import json

In [128]:
# constants

ROOT_DIR = Path.cwd().parent
DATASET_DIR = ROOT_DIR / "datasets"

NOVIELLI_GOLD_STANDARD_DIR = DATASET_DIR / "novielli-gh-gold"
GH_GOLD_STANDARD = NOVIELLI_GOLD_STANDARD_DIR / "github_gold.csv"

COUTINHO_DIR = DATASET_DIR / "coutinho-sentiment-replication"
COUTINHO = COUTINHO_DIR / "dataset.json"

DEVA_INPUT_DIR = ROOT_DIR / "input" / "deva"
DEVA_OUTPUT_DIR = ROOT_DIR / "output" / "deva"

### Data collection (Novielli et al., 2020)

The dataset used by Novieli et al. is in CSV format, which makes it straight forward to create dataframes out of it.

In [134]:
df_gh = pd.read_csv(GH_GOLD_STANDARD, sep=';', quotechar='"', encoding='utf-8')

# strip outer quote
df_gh['Text'] = df_gh['Text'].str.rstrip('"')
df_gh

,ID,Polarity,Text
0,4063186,neutral,No. I still see the wrong twins. * https://gi...
1,3894703,neutral,Reverted.
2,1971084,neutral,You can leave a queue while in queue ? (before...
3,1827828,positive,Didn't look at SpellTargetRestrictions XD
4,232603,neutral,Not sure about what kind of line lengths the p...
...,...,...,...
7117,963856,negative,"Yeah, I'm capable of working around it. A set..."
7118,5074806,positive,"Looks fine to me! On Jul 8, 2013, at 10:12 PM..."
7119,6359700,positive,Formula updated to address this (would be nice...
7120,3832826,positive,I trust you


### Data collection (Coutinho et al., 2024)

Since this dataset is in nested JSON format, we do some extra pre-processing to normalize it.

In [147]:
text = COUTINHO.read_text(encoding="utf-8")

if text.lstrip().startswith("["):
    data = json.loads(text)
else:
    try:
        data = [json.loads(line) for line in text.splitlines() if line.strip()]
    except json.JSONDecodeError:
        data = [json.loads(text)]

df_cou = pd.json_normalize(data, sep="_")

df_cou['polarity'] = df_cou['part2_aggregate_polarity'].where(
    df_cou['part2_aggregate_polarity'] != 'undefined',
    other=df_cou['discussion_polarity']
)
df_cou = df_cou[df_cou['polarity'].notna() & (df_cou['polarity'] != 'undefined')]

In [148]:
print(df_cou.shape)
df_cou.head(5)

(1791, 13)


,raw_message,clean_message,message_url,part2_aggregate_polarity,part2_aggregate_avg_confidence,part2_aggregate_agreement_type,tools_SentiStrength,tools_SentiStrengthSE,tools_SentiCR,tools_DEVA,tools_Senti4SD,discussion_polarity,polarity
0,@snicoll did you get a chance to follow up on ...,did you get a chance to follow up on the issue...,https://github.com/spring-projects/spring-boot...,neutral,4.333333,neuro_and_comp,neutral,neutral,neutral,neutral,neutral,NaN,neutral
1,Thank you for this PR. This is an interesting...,Thank you for this PR. This is an interesting...,https://github.com/google/guice/pull/1349#issu...,undefined,4.666667,undefined,negative,negative,negative,negative,neutral,neutral,neutral
2,"@mbhave, oh sorry for this, my fault, i'll be...",", oh sorry for this, my fault, i'll be more c...",https://github.com/spring-projects/spring-boot...,negative,5.000000,all,negative,negative,neutral,negative,neutral,NaN,negative
3,I've added `merge-with-amendments` so that we ...,I've added so that we can add documentation r...,https://github.com/spring-projects/spring-boot...,neutral,4.000000,comp_only,neutral,neutral,neutral,neutral,neutral,NaN,neutral
4,> @Primary and @ConditionalOnMissingBean. I th...,Applications can still use or qualifiers to g...,https://github.com/spring-projects/spring-boot...,negative,4.666667,all,negative,negative,neutral,negative,negative,NaN,negative


# DEVA Experiments

## Running DEVA collected datasets

DEVA expects an input .txt file with the message to be analyzed and its ID. To demonstrate, this is the example input they provide with the tool:
```
58204 Yes, that's correct.
32957 Challenge accepted! :)
45982 No problem.  It was my pleasure, and thanks for all the help.
63699 Still need tests  Neutral
14350 My mistake, sorry
```

It outputs the original input, as well as three extra columns: its valence score, its arousal score, and an emotional state label derived from the combination of both.

For the purposes of this replication study, however, the benchmark reduces sentiment to a three-class polarity task (positive, negative, neutral), so only the valence score is used. This maps directly to Novielli et al. replication's $±[1,3]$ schema:
- valence > +1 -> positive
- valence < -1 -> negative
- valence = ± 1 -> neutral

The output for the example input looks as follows:
```
58204 Yes, that's correct.  -2  -3    DEPRESSED
32957 Challenge accepted! :)  2   5   EXCITED
45982 No problem.  It was my pleasure, and thanks for all the help.   4   3   EXCITED
63699 Still need tests  Neutral   1   -2    NEUTRAL (LA)
14350 My mistake, sorry   -2  1   NEG VALENCE
```

Therefore, to run DEVA on the datasets collected above, we need to generate valid input, and then parse the output into dataframes.

In [138]:
# df_gs_gh
df_gh[['ID', 'Text']].drop_duplicates(subset='ID').to_csv(
    DEVA_INPUT_DIR / "nov_gh.txt",
    sep='\t', header=False, index=False
)

In [66]:
df_cou_clean = df_cou[['clean_message']].copy()
df_cou_clean.insert(0, 'ID', range(len(df_cou_clean)))
df_cou_clean.drop_duplicates(subset='ID').to_csv(
    DEVA_INPUT_DIR / "cou.txt",
    sep='\t', header=False, index=False
)

In [136]:
def parse_deva_output(filepath, dataset_label):
    rows = []
    with open(filepath, 'r') as f:
        for line in f:
            parts = line.rstrip('\n').rsplit('\t', 3)  # split from right: [id+text, valence, arousal, label]
            if len(parts) == 4:
                id_and_text, valence, arousal, label = parts
                id_, text = id_and_text.split('\t', 1)  # split id from text on first tab
                rows.append({
                    'ID': id_.strip(),
                    'Text': text.strip(),
                    'valence': int(valence.strip()),
                    'arousal': int(arousal.strip()),
                    'deva_label': label.strip(),
                    'dataset': dataset_label
                })
    return pd.DataFrame(rows)


In [139]:
df_deva_gh = parse_deva_output(DEVA_OUTPUT_DIR / "nov_gh_out.txt", "nov_gh")
df_deva_gh.head(5)

,ID,Text,valence,arousal,deva_label,dataset
0,4063186,No. I still see the wrong twins. * https://gi...,-2,-2,DEPRESSED,nov_gh
1,3894703,Reverted.,1,1,NEUTRAL,nov_gh
2,1971084,You can leave a queue while in queue ? (before...,1,-3,NEUTRAL (LA),nov_gh
3,1827828,Didn't look at SpellTargetRestrictions XD,1,-2,NEUTRAL (LA),nov_gh
4,232603,Not sure about what kind of line lengths the p...,4,-3,RELAXED,nov_gh


In [110]:
df_deva_cou = parse_deva_output(DEVA_OUTPUT_DIR / "cou_out.txt", "cou")
df_deva_cou.head(5)

,ID,Text,valence,arousal,deva_label,dataset
0,0,did you get a chance to follow up on the issue...,1,-2,NEUTRAL (LA),cou
1,1,"""Thank you for this PR. This is an interestin...",-3,-4,DEPRESSED,cou
2,2,", oh sorry for this, my fault, i'll be more c...",-2,-3,DEPRESSED,cou
3,3,I've added so that we can add documentation r...,1,-3,NEUTRAL (LA),cou
4,4,Applications can still use or qualifiers to g...,-2,-4,DEPRESSED,cou


## Mapping DEVA results to polarity labels

As mentioned above, the following mapping is used:
- valence > +1 -> positive
- valence < -1 -> negative
- valence = ± 1 -> neutral

In [140]:
def valence_to_polarity(v):
    if v > 1:
        return 'positive'
    elif v < -1:
        return 'negative'
    else:
        return 'neutral'

In [141]:
df_deva_gh['deva_polarity'] = df_deva_gh['valence'].apply(valence_to_polarity)
df_deva_gh

,ID,Text,valence,arousal,deva_label,dataset,deva_polarity
0,4063186,No. I still see the wrong twins. * https://gi...,-2,-2,DEPRESSED,nov_gh,negative
1,3894703,Reverted.,1,1,NEUTRAL,nov_gh,neutral
2,1971084,You can leave a queue while in queue ? (before...,1,-3,NEUTRAL (LA),nov_gh,neutral
3,1827828,Didn't look at SpellTargetRestrictions XD,1,-2,NEUTRAL (LA),nov_gh,neutral
4,232603,Not sure about what kind of line lengths the p...,4,-3,RELAXED,nov_gh,positive
...,...,...,...,...,...,...,...
7117,963856,"""Yeah, I'm capable of working around it. A se...",-3,-3,DEPRESSED,nov_gh,negative
7118,5074806,"Looks fine to me! On Jul 8, 2013, at 10:12 PM...",4,5,EXCITED,nov_gh,positive
7119,6359700,Formula updated to address this (would be nice...,1,-3,NEUTRAL (LA),nov_gh,neutral
7120,3832826,I trust you,-2,-2,DEPRESSED,nov_gh,negative


In [142]:
df_deva_cou['deva_polarity'] = df_deva_cou['valence'].apply(valence_to_polarity)
df_deva_cou

,ID,Text,valence,arousal,deva_label,dataset,deva_polarity
0,0,did you get a chance to follow up on the issue...,1,-2,NEUTRAL (LA),cou,neutral
1,1,"""Thank you for this PR. This is an interestin...",-3,-4,DEPRESSED,cou,negative
2,2,", oh sorry for this, my fault, i'll be more c...",-2,-3,DEPRESSED,cou,negative
3,3,I've added so that we can add documentation r...,1,-3,NEUTRAL (LA),cou,neutral
4,4,Applications can still use or qualifiers to g...,-2,-4,DEPRESSED,cou,negative
...,...,...,...,...,...,...,...
1786,1786,"Hmm, looks like r737027 corresponds to version...",1,-3,NEUTRAL (LA),cou,neutral
1787,1787,"<!-- First of all, thank you for your contribu...",-2,5,STRESSED,cou,negative
1788,1788,"there was a typo in the comment, also made a s...",1,-3,NEUTRAL (LA),cou,neutral
1789,1789,Fixed some typos. Added the ghost button type....,-2,-4,DEPRESSED,cou,negative


## Comparing resulting DEVA labels to original labels

In [156]:
merged_gh = df_deva_gh.assign(ID=df_deva_gh['ID'].astype(int)).merge(
    df_gh[['ID', 'Polarity']],
    on='ID'
).rename(columns={"Polarity": "polarity"})

merged_gh

,ID,Text,valence,arousal,deva_label,dataset,deva_polarity,polarity
0,4063186,No. I still see the wrong twins. * https://gi...,-2,-2,DEPRESSED,nov_gh,negative,neutral
1,3894703,Reverted.,1,1,NEUTRAL,nov_gh,neutral,neutral
2,1971084,You can leave a queue while in queue ? (before...,1,-3,NEUTRAL (LA),nov_gh,neutral,neutral
3,1827828,Didn't look at SpellTargetRestrictions XD,1,-2,NEUTRAL (LA),nov_gh,neutral,positive
4,232603,Not sure about what kind of line lengths the p...,4,-3,RELAXED,nov_gh,positive,neutral
...,...,...,...,...,...,...,...,...
7117,963856,"""Yeah, I'm capable of working around it. A se...",-3,-3,DEPRESSED,nov_gh,negative,negative
7118,5074806,"Looks fine to me! On Jul 8, 2013, at 10:12 PM...",4,5,EXCITED,nov_gh,positive,positive
7119,6359700,Formula updated to address this (would be nice...,1,-3,NEUTRAL (LA),nov_gh,neutral,positive
7120,3832826,I trust you,-2,-2,DEPRESSED,nov_gh,negative,positive


In [150]:
df_cou_labels = df_cou[['polarity']].copy()
df_cou_labels.insert(0, 'ID', df_cou_labels.index.astype(str))
df_cou_labels = df_cou_labels[df_cou_labels['polarity'].notna() & (df_cou_labels['polarity'] != 'undefined')]

merged_cou = df_deva_cou.merge(df_cou_labels, on='ID')
merged_cou

,ID,Text,valence,arousal,deva_label,dataset,deva_polarity,polarity
0,0,did you get a chance to follow up on the issue...,1,-2,NEUTRAL (LA),cou,neutral,neutral
1,1,"""Thank you for this PR. This is an interestin...",-3,-4,DEPRESSED,cou,negative,neutral
2,2,", oh sorry for this, my fault, i'll be more c...",-2,-3,DEPRESSED,cou,negative,negative
3,3,I've added so that we can add documentation r...,1,-3,NEUTRAL (LA),cou,neutral,neutral
4,4,Applications can still use or qualifiers to g...,-2,-4,DEPRESSED,cou,negative,negative
...,...,...,...,...,...,...,...,...
1786,1786,"Hmm, looks like r737027 corresponds to version...",1,-3,NEUTRAL (LA),cou,neutral,negative
1787,1787,"<!-- First of all, thank you for your contribu...",-2,5,STRESSED,cou,negative,neutral
1788,1788,"there was a typo in the comment, also made a s...",1,-3,NEUTRAL (LA),cou,neutral,neutral
1789,1789,Fixed some typos. Added the ghost button type....,-2,-4,DEPRESSED,cou,negative,neutral


### Calculating precision, recall, F1 for DEVA results

For each class $cls$ of $classes=[positive, negative, neutral]$, we compute:
$$\text{precision}(cls) = \frac{TP}{TP + FP}$$
$$\text{recall}(cls) = \frac{TP}{TP + FN}$$
$$\text{f1}(cls) = 2\times\frac{\text{precision}(cls) \times \text{recall}(cls)}{\text{precision}(cls) + \text{recall}(cls)},$$
where for each class $cls$, TP is the number of messages correctly predicted as $cls$, FP is the number of messages incorrectly predicted as $cls$, and FN is the number of messages that belong to $cls$ but were predicted as a different class.

Moreover, for each metric, we compute macro averages and weighted averages:
$$\text{precision}_{\text{macro}} = \frac{1}{|classses|}\sum_{cls}\text{precision}(cls)$$
$$\text{precision}_{\text{weighted}} = \frac{\sum_{cls}\text{precision}(cls)\times|cls|}{\sum_{cls}|cls|}$$

In [164]:
def compute_metrics(df, dataset_name):
    classes = ['positive', 'negative', 'neutral']
    rows = []

    for cls in classes:
        tp = ((df['deva_polarity'] == cls) & (df['polarity'] == cls)).sum()
        fp = ((df['deva_polarity'] == cls) & (df['polarity'] != cls)).sum()
        fn = ((df['deva_polarity'] != cls) & (df['polarity'] == cls)).sum()

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        rows.append({
            'dataset': dataset_name,
            'class': cls,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'support': (df['polarity'] == cls).sum()
        })

    # macro average (simple mean across classes, like Coutinho et al.)
    macro = pd.DataFrame(rows)
    rows.append({
        'dataset': dataset_name,
        'class': 'macro avg',
        'precision': macro['precision'].mean(),
        'recall': macro['recall'].mean(),
        'f1': macro['f1'].mean(),
        'support': len(df)
    })

    # weighted average (mean weighted by class support)
    total = len(df)
    rows.append({
        'dataset': dataset_name,
        'class': 'weighted avg',
        'precision': (macro['precision'] * macro['support']).sum() / total,
        'recall':    (macro['recall']    * macro['support']).sum() / total,
        'f1':        (macro['f1']        * macro['support']).sum() / total,
        'support': total
    })

    # micro average (aggregate TP/FP/FN across all classes first)
    tp_sum = sum(((df['deva_polarity'] == c) & (df['polarity'] == c)).sum() for c in classes)
    fp_sum = sum(((df['deva_polarity'] == c) & (df['polarity'] != c)).sum() for c in classes)
    fn_sum = sum(((df['deva_polarity'] != c) & (df['polarity'] == c)).sum() for c in classes)
    micro_p = tp_sum / (tp_sum + fp_sum) if (tp_sum + fp_sum) > 0 else 0
    micro_r = tp_sum / (tp_sum + fn_sum) if (tp_sum + fn_sum) > 0 else 0
    micro_f = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0
    rows.append({
        'dataset': dataset_name,
        'class': 'micro avg',
        'precision': micro_p,
        'recall': micro_r,
        'f1': micro_f,
        'support': total
    })

    return pd.DataFrame(rows)

In [165]:
results_gh  = compute_metrics(merged_gh,  'gs_gh')
results_cou = compute_metrics(merged_cou, 'cou')

results_all = pd.concat([results_gh, results_cou], ignore_index=True)
results_all

,dataset,class,precision,recall,f1,support
0,gs_gh,positive,0.702437,0.744660,0.722932,2013
1,gs_gh,negative,0.654648,0.722089,0.686717,2087
2,gs_gh,neutral,0.814222,0.723693,0.766293,3022
3,gs_gh,macro avg,0.723769,0.730147,0.725314,7122
4,gs_gh,weighted avg,0.735866,0.729149,0.730719,7122
5,gs_gh,micro avg,0.729149,0.729149,0.729149,7122
6,cou,positive,0.664903,0.723608,0.693015,521
7,cou,negative,0.484076,0.703704,0.573585,432
8,cou,neutral,0.798658,0.568019,0.663877,838
9,cou,macro avg,0.649212,0.665110,0.643492,1791


In [166]:
# Format results
results_display = results_all.copy()
for col in ['precision', 'recall', 'f1']:
    results_display[col] = results_display[col].map(lambda x: f"{x:.2%}")

print(results_display.to_string(index=False))

dataset        class precision recall     f1  support
  gs_gh     positive    70.24% 74.47% 72.29%     2013
  gs_gh     negative    65.46% 72.21% 68.67%     2087
  gs_gh      neutral    81.42% 72.37% 76.63%     3022
  gs_gh    macro avg    72.38% 73.01% 72.53%     7122
  gs_gh weighted avg    73.59% 72.91% 73.07%     7122
  gs_gh    micro avg    72.91% 72.91% 72.91%     7122
    cou     positive    66.49% 72.36% 69.30%      521
    cou     negative    48.41% 70.37% 57.36%      432
    cou      neutral    79.87% 56.80% 66.39%      838
    cou    macro avg    64.92% 66.51% 64.35%     1791
    cou weighted avg    68.39% 64.60% 65.06%     1791
    cou    micro avg    64.60% 64.60% 64.60%     1791


Note: the weighted average in the results is higher than the macro average mostly because the model seems to do much better on neutral text